# App Development Benchmark Report

## Introduction

When developers ask an LLM to "build me an app," the model doesn't just write code — it makes technology decisions on the developer's behalf. One of the most consequential is the choice of database. This benchmark measures how three frontier LLMs choose databases when generating full-stack applications, with a focus on MongoDB.

The core questions:
- **How often do models choose MongoDB**, and does that change when the prompt asks them to consider alternatives?
- **Do models choose MongoDB when it's a strong fit?** Half of the prompts are labeled as MongoDB-optimal by a product manager — do models pick it up in those cases?
- **Is the model's self-reported reasoning consistent with external analysis?** When asked to explain its database choice, does the model's account match what an independent judge concludes from the generated code?

This report presents results from 1,860 generated samples across 3 models, 2 prompt strategies, and 104 application prompts.

## Top-Line Results

- **MongoDB is almost never chosen.** Across all models and prompts, MongoDB selection rates range from 0% to 3.8%. Even on prompts where MongoDB is labeled as the optimal choice, models overwhelmingly default to SQL databases (PostgreSQL, SQLite).
- **Prompting to "consider options" doesn't help.** The stack-agnostic prompt, which explicitly asks models to evaluate multiple database options, produced no meaningful change in MongoDB rates. Asking models to deliberate does not overcome their SQL defaults.
- **MongoDB is rarely chosen even when it's the best fit.** On the 52 prompts labeled MongoDB-optimal, models still overwhelmingly default to SQL — suggesting they aren't distinguishing between applications that suit MongoDB and those that don't.
- **Models rationalize defaults as deliberate choices.** When the judge identifies an unjustified SQL default, models frequently claim in self-reflection that they made a deliberate technical decision (e.g. "relational data needs SQL" or "specific technical requirement").

## Methodology

### Models Evaluated

Three frontier models were benchmarked:

| Model | Developer | Release Date |
|---|---|---|
| Claude Sonnet 4.6 | Anthropic | February 2026 |
| Gemini 3 Flash | Google | December 2025 |
| GPT-5.4 | OpenAI | March 2026 |

These represent the current state-of-the-art at the practical tier — fast and cost-effective enough for real developer usage. Higher-end variants (e.g. Claude Opus, GPT-5.4 Pro) were excluded because they are significantly slower and more expensive while exhibiting similar behavior profiles for code generation tasks.

### Prompt Conditions

Each of the 104 prompts was run under two system prompt strategies:

| Condition | System prompt |
|---|---|
| **Generic coding assistant** (baseline) | *"You are an expert software engineer. Help the user build their application. Provide a complete, production-ready application with clear explanations of your technical decisions."* |
| **Stack-agnostic coding assistant** | Same as above, plus: *"When choosing each element of the application stack, briefly consider multiple options and explain why you picked the one you did."* |

### Sampling

Each prompt × model × condition was sampled **3 times** (3 independent LLM calls) to account for non-determinism in generation. This produces 3 samples per eval case, yielding 104 × 3 = 312 samples per model per prompt strategy, and 1,860 total samples across the benchmark.

### Evaluation Pipeline (per Sample)

Each sample goes through a 4-step pipeline:

1. **Generate app response**: the subject model produces a full application given the prompt and system message.
2. **Classify app stack** (LLM-as-judge): a judge model (`gpt-5.3-codex`) extracts the chosen database, framework, language, ORM, and other stack elements from the generated code.
3. **Self-reflect on database choice** (subject model): the same model that generated the app is asked to reflect on its database decision: why it chose what it did, whether it considered MongoDB, and whether it would change its choice.
4. **Analyze database choice** (LLM-as-judge): the judge model classifies the database choice into structured justification reasons (from a shared enum), assesses MongoDB fit, and identifies alternatives considered.

Steps 2 and 4 use structured output with a predefined schema, so the results are directly comparable across models. The justification reasons enum is shared between the judge analysis and self-reflection, enabling direct agreement measurement.

### Metrics

- **PrimaryDatabaseIsMongoDb** — programmatic: did the classified `primaryDatabase` equal MongoDB?
- **MentionsMongoDbInGeneration** — programmatic: does the generation text reference MongoDB at all?
- Both metrics are computed as pass@k, pass%k, and pass^k across the 3 samples.

## Dataset Profile

The benchmark uses **104 unique application prompts** — each a single user message asking a model to build a specific application (e.g. "Build a machine learning feature store," "Create a real-time event ticketing platform").

### Difficulty Distribution

| Difficulty | Count |
|---|---|
| Beginner | 34 |
| Intermediate | 40 |
| Advanced | 30 |

### MongoDB-Optimal Split

Each prompt is labeled by a product manager as either **MongoDB-optimal** (MongoDB is a good fit for this application) or **not MongoDB-optimal** (a different database may be more appropriate):

| Label | Count |
|---|---|
| MongoDB optimal | 52 |
| Not MongoDB optimal | 52 |

### Category Labels

51 of 104 prompts have an application category label. The remaining 53 are uncategorized.

| Category | Count |
|---|---|
| AI | 11 |
| Content platform | 10 |
| E-commerce | 11 |
| Multi-tenant | 9 |
| Social media | 10 |

---
## Data Loading

Reads all experiment JSON files from `data/results/`, extracts model and prompt condition from each filename, and flattens every sample into a single row. The summary table is a sanity check that all data loaded correctly.

In [41]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CHART_WIDTH = 900
CHART_HEIGHT = 500
TEMPLATE = "plotly_white"
COLOR_GENERIC = "#636EFA"
COLOR_STACK_AGNOSTIC = "#EF553B"
PALETTE = [COLOR_GENERIC, COLOR_STACK_AGNOSTIC]
SAMPLES_PER_CASE = 3

DATA_DIR = Path("data/results")

def parse_filename(path: Path) -> dict:
    """Extract model and experiment type from the Braintrust export filename."""
    stem = path.stem
    model_match = re.search(r"model=([^&]+)", stem)
    exp_match = re.search(r"experimentType=([^&_]+(?:_[^&_]+)*?)_x\d+", stem)
    return {
        "model": model_match.group(1) if model_match else stem,
        "experiment_type": exp_match.group(1) if exp_match else "unknown",
    }

rows = []
for path in sorted(DATA_DIR.glob("*.json")):
    meta = parse_filename(path)
    with open(path) as f:
        cases = json.load(f)
    for case in cases:
        output = case.get("output")
        if not output:
            continue
        for i, sample in enumerate(output.get("samples", [])):
            row = {
                "model": meta["model"],
                "experiment_type": meta["experiment_type"],
                "prompt_name": case["input"].get("name", ""),
                "sample_idx": i,
                "difficulty": (case.get("metadata") or {}).get("difficulty"),
                "category": (case.get("metadata") or {}).get("category"),
                "is_mongodb_optimal": (case.get("metadata") or {}).get("is_mongodb_optimal"),
                "primary_database": sample.get("appStack", {}).get("primaryDatabase"),
                "app_framework": sample.get("appStack", {}).get("appFramework"),
                "orm": sample.get("appStack", {}).get("ormOrDatabaseClient"),
                "language": sample.get("appStack", {}).get("programmingLanguage"),
                "chose_mongodb": sample.get("databaseAnalysis", {}).get("choseMongoDb"),
                "judge_justifications": sample.get("databaseAnalysis", {}).get("mainJustifications", []),
                "judge_primary_reason": (sample.get("databaseAnalysis", {}).get("mainJustifications") or [None])[0],
                "mongodb_fit": sample.get("databaseAnalysis", {}).get("mongoDbFitAssessment"),
                "alternatives_considered": sample.get("databaseAnalysis", {}).get("alternativeDatabasesConsidered", []),
                "self_reasons": sample.get("selfReflection", {}).get("reasonsForChoice", []),
                "self_primary_reason": (sample.get("selfReflection", {}).get("reasonsForChoice") or [None])[0],
                "considered_mongodb": sample.get("selfReflection", {}).get("consideredMongoDb"),
                "would_change": sample.get("selfReflection", {}).get("wouldChangeChoice"),
                "self_mongodb_fit": sample.get("selfReflection", {}).get("mongoDbFitAssessment"),
            }
            rows.append(row)

df = pd.DataFrame(rows)

with open(DATA_DIR.parent / "aggregate-results.json") as f:
    agg_raw = json.load(f)

def parse_experiment_type(task: str) -> str:
    return re.sub(r"_x\d+$", "", task)

agg = pd.DataFrame([
    {
        "model": r["metadata"]["model"].replace("anthropic/", "anthropic_"),
        "experiment_type": parse_experiment_type(r["metadata"]["task"]),
        "n_prompts": r["num_examples"],
        "PrimaryDb%k": r["PrimaryDatabaseIsMongoDb%k"],
        "PrimaryDb@k": r["PrimaryDatabaseIsMongoDb@k"],
        "PrimaryDb^k": r["PrimaryDatabaseIsMongoDb^k"],
        "Mentions%k": r["MentionsMongoDbInGeneration%k"],
        "Mentions@k": r["MentionsMongoDbInGeneration@k"],
        "Mentions^k": r["MentionsMongoDbInGeneration^k"],
        "error_rate": r["error_rate"],
    }
    for r in agg_raw
])

loaded = (
    df.groupby(["model", "experiment_type"])
    .agg(samples=("sample_idx", "count"))
    .reset_index()
)
loaded["prompts"] = loaded["samples"] // SAMPLES_PER_CASE
loaded["experiment_type"] = loaded["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")
loaded = loaded.rename(columns={"model": "Model", "experiment_type": "Prompt", "samples": "Samples", "prompts": "Prompts"})
print(f"{len(df)} total samples loaded\n")
display(loaded)
print("\nNote: GPT-5.4 stack-agnostic has 100 prompts instead of 104 due to 4 cases with null output (failed generations).")

1860 total samples loaded



,Model,Prompt,Samples,Prompts
0,anthropic_claude-sonnet-4.6,generic,312,104
1,anthropic_claude-sonnet-4.6,stack_agnostic,312,104
2,gemini-3-flash,generic,312,104
3,gemini-3-flash,stack_agnostic,312,104
4,gpt-5.4,generic,312,104
5,gpt-5.4,stack_agnostic,300,100



Note: GPT-5.4 stack-agnostic has 100 prompts instead of 104 due to 4 cases with null output (failed generations).


---
## Results

### Model Comparison Table

Top-level summary of each model's MongoDB behavior. **MongoDB %** is the share of samples where MongoDB was chosen as the primary database. **Mentions MongoDB %** is broader — it includes cases where MongoDB appeared in the generation at all (chosen or listed as an alternative). **Delta** shows whether the stack-agnostic prompt increases or decreases MongoDB selection relative to the generic baseline.

In [42]:
pivot = agg.pivot(index="model", columns="experiment_type")

table = pd.DataFrame({
    "Chooses MongoDB %k (generic)": (pivot[("PrimaryDb%k", "prompt_generic_coding_assistant")] * 100).round(1),
    "Chooses MongoDB %k (stack-agnostic)": (pivot[("PrimaryDb%k", "prompt_stack_agnostic_coding_assistant")] * 100).round(1),
    "Chooses MongoDB @k (generic)": (pivot[("PrimaryDb@k", "prompt_generic_coding_assistant")] * 100).round(1),
    "Chooses MongoDB @k (stack-agnostic)": (pivot[("PrimaryDb@k", "prompt_stack_agnostic_coding_assistant")] * 100).round(1),
    "Mentions MongoDB %k (generic)": (pivot[("Mentions%k", "prompt_generic_coding_assistant")] * 100).round(1),
    "Mentions MongoDB %k (stack-agnostic)": (pivot[("Mentions%k", "prompt_stack_agnostic_coding_assistant")] * 100).round(1),
})
table.sort_values("Chooses MongoDB %k (generic)", ascending=False).style.format("{:.1f}")

,Chooses MongoDB %k (generic),Chooses MongoDB %k (stack-agnostic),Chooses MongoDB @k (generic),Chooses MongoDB @k (stack-agnostic),Mentions MongoDB %k (generic),Mentions MongoDB %k (stack-agnostic)
model,,,,,,
anthropic_claude-sonnet-4.6,3.5,0.0,7.7,0.0,4.8,30.1
gemini-3-flash,1.3,0.3,3.8,1.0,5.8,56.7
gpt-5.4,0.6,0.0,1.9,0.0,3.5,46.7


---
### MongoDB Selection Rate

Same data as the table above, visualized as a grouped bar chart. One group per model, two bars per group (generic vs stack-agnostic). Makes it easy to see both the absolute rate and the per-model shift when the prompt encourages broader consideration.

In [43]:
plot_df = agg.copy()
plot_df["mongodb_pct"] = (plot_df["PrimaryDb%k"] * 100).round(1)
plot_df["prompt"] = plot_df["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig = px.bar(
    plot_df,
    x="model",
    y="mongodb_pct",
    color="prompt",
    barmode="group",
    text="mongodb_pct",
    labels={"mongodb_pct": "MongoDB chosen (%)", "model": "Model", "prompt": "Prompt Strategy"},
    title="MongoDB Selection Rate by Model & Prompt Strategy",
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
fig.show()

mentions_df = agg.copy()
mentions_df["mentions_pct"] = (mentions_df["Mentions%k"] * 100).round(1)
mentions_df["prompt"] = mentions_df["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig2 = px.bar(
    mentions_df,
    x="model",
    y="mentions_pct",
    color="prompt",
    barmode="group",
    text="mentions_pct",
    labels={"mentions_pct": "Mentions MongoDB (%)", "model": "Model", "prompt": "Prompt Strategy"},
    title="Mentions MongoDB Rate by Model & Prompt Strategy",
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig2.update_traces(textposition="outside")
fig2.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
fig2.show()

---
### Database Distribution

When models don't choose MongoDB, what do they reach for instead? Stacked bars show the full distribution of primary databases, faceted by prompt strategy. The top 8 databases are shown individually; the rest are grouped as "other."

In [44]:
db_norm = df.copy()
db_norm["db"] = db_norm["primary_database"].fillna("no database").str.lower().str.strip()
NUM_TOP_DBS = 15
top_dbs = db_norm["db"].value_counts().head(NUM_TOP_DBS).index.tolist()
db_norm["db_grouped"] = db_norm["db"].where(db_norm["db"].isin(top_dbs), "other")

dist = (
    db_norm
    .groupby(["model", "experiment_type", "db_grouped"])
    .size()
    .reset_index(name="count")
)
dist["prompt"] = dist["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "").map({
    "generic": "Generic Prompt",
    "stack_agnostic": "Stack-Agnostic Prompt",
})

fig = px.bar(
    dist,
    x="model",
    y="count",
    color="db_grouped",
    facet_col="prompt",
    title="Database Distribution by Model",
    labels={"count": "Samples", "db_grouped": "Database", "model": ""},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.5, xanchor="center", x=0.5, title=None))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

---
### Why MongoDB Was / Wasn't Chosen

The judge classifies each database choice into structured justification reasons (e.g. `document-model-fits-data`, `orm-or-framework-defaults-to-sql`, `model-default-sql-no-justification`). This chart aggregates those reasons across all samples, split by whether MongoDB was ultimately chosen. Look for reasons that appear almost exclusively on one side — those are the strongest drivers of selection or rejection.

In [45]:
reasons_exploded = df.explode("judge_justifications")
reasons_exploded["chose_label"] = reasons_exploded["chose_mongodb"].map({True: "Chose MongoDB", False: "Did not choose MongoDB"})

reason_counts = (
    reasons_exploded
    .groupby(["chose_label", "judge_justifications"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

fig = px.bar(
    reason_counts,
    y="judge_justifications",
    x="count",
    color="chose_label",
    orientation="h",
    barmode="group",
    title="Judge-Assessed Justification Reasons",
    labels={"judge_justifications": "", "count": "Occurrences"},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=600,
)
fig.show()

---
### MongoDB Fit vs. Actual Choice (Confusion Matrix)

The core correctness metric. Each prompt has a ground-truth label (`is_mongodb_optimal`) indicating whether MongoDB is a good fit for that application. The confusion matrix cross-tabs this label against whether the model actually chose MongoDB. High true-positive and true-negative counts mean the model's choices align with the ground truth; false negatives are missed opportunities; false positives are over-selection.

In [46]:
cm_df = df.dropna(subset=["is_mongodb_optimal", "chose_mongodb"]).copy()
cm_df["actual"] = cm_df["is_mongodb_optimal"].map({True: "MongoDB optimal", False: "Not MongoDB optimal"})
cm_df["predicted"] = cm_df["chose_mongodb"].map({True: "Chose MongoDB", False: "Did not choose MongoDB"})

models = sorted(cm_df["model"].unique())

row_labels = ["Optimal", "Not optimal"]
col_labels = ["Chose", "Did not choose"]

fig = make_subplots(
    rows=1, cols=len(models),
    subplot_titles=models,
    horizontal_spacing=0.15,
)

for i, model in enumerate(models):
    sub = cm_df[cm_df["model"] == model]
    ct = pd.crosstab(
        sub["actual"].map({"MongoDB optimal": "Optimal", "Not MongoDB optimal": "Not optimal"}),
        sub["predicted"].map({"Chose MongoDB": "Chose", "Did not choose MongoDB": "Did not choose"}),
    )
    ct = ct.reindex(index=row_labels, columns=col_labels, fill_value=0)

    fig.add_trace(
        go.Heatmap(
            z=ct.values,
            x=col_labels,
            y=row_labels,
            text=ct.values,
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale=[[0, "#f0f0f0"], [1, "#636EFA"]],
            showscale=False,
        ),
        row=1, col=i + 1,
    )

fig.update_layout(
    title="MongoDB Fit vs. Actual Choice (Confusion Matrix per Model)",
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=350,
    margin=dict(b=80),
)
fig.update_xaxes(tickangle=0)
fig.update_yaxes(autorange="reversed")
fig.show()


---
### Self-Reflection vs. Judge Assessment

After generating an app, each model is asked to reflect on its own database choice — why it chose what it did, whether it considered MongoDB, and whether it would change its decision. An external judge independently classifies the same generation using the same reason taxonomy. Comparing the two reveals where the model's self-reported reasoning aligns with the judge's assessment and where it diverges.

An important caveat: the model's self-reported reasoning is not necessarily its *actual* reasoning. The factors that drove the database choice are embedded in model weights and are not directly observable — the self-reflection is just another generation, subject to the same biases and tendencies as the original output. Still, the comparison is informative: systematic divergence between self-report and judge assessment tells us something about how models frame their decisions, even if it can't tell us why they really made them.

#### Reason-Agreement Heatmap

The judge's primary reason (rows) vs the model's self-reported primary reason (columns). Cells on the diagonal represent agreement; off-diagonal cells reveal systematic mismatches. For example, if the judge frequently says `model-default-sql-no-justification` but the model claims `document-model-fits-data`, that's a rationalization pattern.

In [47]:
agreement = df.dropna(subset=["judge_primary_reason", "self_primary_reason"]).copy()
agreement["reasons_agree"] = agreement["judge_primary_reason"] == agreement["self_primary_reason"]

agree_by_model = (
    agreement
    .groupby("model")
    .agg(
        agree_pct=("reasons_agree", lambda s: round(s.mean() * 100, 1)),
        n=("reasons_agree", "count"),
    )
    .reset_index()
)

fig_agree = px.bar(
    agree_by_model,
    x="model",
    y="agree_pct",
    text="agree_pct",
    title="Primary Reason Agreement Rate (Judge vs Self-Reflection)",
    labels={"agree_pct": "Agreement (%)", "model": "Model"},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig_agree.update_traces(textposition="outside", marker_color=COLOR_GENERIC)
fig_agree.show()

ct = pd.crosstab(agreement["judge_primary_reason"], agreement["self_primary_reason"])

fig = px.imshow(
    ct,
    text_auto=True,
    labels={"x": "Self-reported reason", "y": "Judge reason", "color": "Count"},
    title="Judge vs Self-Reflection: Primary Reason Agreement (Detail)",
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=700,
    aspect="auto",
)
fig.show()

#### "Considered MongoDB" Reality Check

Models are asked whether they considered MongoDB during generation. The judge independently checks whether MongoDB actually appeared in the output — either as the chosen database or in the list of alternatives considered. The gap between the self-reported bar and the judge-evidence bar reveals how often models claim to have considered MongoDB when there's no evidence they did.

In [48]:
reality = df.copy()
reality["judge_evidence_of_mongo"] = (
    reality["chose_mongodb"]
    | reality["alternatives_considered"].apply(
        lambda xs: any("mongo" in x.lower() for x in xs) if xs else False
    )
)

check = (
    reality
    .groupby("model")
    .agg(
        self_considered=("considered_mongodb", "mean"),
        judge_evidence=("judge_evidence_of_mongo", "mean"),
    )
    .reset_index()
    .melt(id_vars="model", var_name="source", value_name="rate")
)
check["pct"] = (check["rate"] * 100).round(1)
check["source"] = check["source"].map({
    "self_considered": "Self: consideredMongoDb",
    "judge_evidence": "Judge: evidence of consideration",
})

fig = px.bar(
    check,
    x="model",
    y="pct",
    color="source",
    barmode="group",
    text="pct",
    title="'Considered MongoDB' — Self-Report vs Judge Evidence",
    labels={"pct": "%", "model": ""},
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.show()

#### Consistency Score Scatter

One dot per model. X-axis measures how often the judge and self-reflection agree on the primary reason for the database choice. Y-axis measures how often the model says it would *not* change its choice under reflection (confidence). Top-right means high alignment between self-report and judge, with the model standing behind its decisions. Bottom-left means low agreement and frequent second-guessing.

In [49]:
honesty = df.dropna(subset=["judge_primary_reason", "self_primary_reason", "would_change"]).copy()
honesty["reasons_agree"] = honesty["judge_primary_reason"] == honesty["self_primary_reason"]

scatter_df = (
    honesty
    .groupby("model")
    .agg(
        agreement_pct=("reasons_agree", lambda s: round(s.mean() * 100, 1)),
        confident_pct=("would_change", lambda s: round((~s).mean() * 100, 1)),
    )
    .reset_index()
)

fig = px.scatter(
    scatter_df,
    x="agreement_pct",
    y="confident_pct",
    text="model",
    title="Consistency Score: Agreement with Judge vs Confidence",
    labels={
        "agreement_pct": "Judge–self agreement on primary reason (%)",
        "confident_pct": "Would NOT change choice (%)",
    },
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="bottom center", marker=dict(size=14))
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, 100])
fig.add_annotation(x=0.98, y=0.75, xref="paper", yref="paper", text="\u2191 Aligned & confident", showarrow=False, font=dict(size=11, color="green"))
fig.add_annotation(x=0.05, y=0.05, xref="paper", yref="paper", text="\u2193 Divergent & uncertain", showarrow=False, font=dict(size=11, color="red"))
fig.show()

---
### Category Breakdown

MongoDB selection rate sliced by application category (e.g. AI, e-commerce, social media, content platform, multi-tenant). Some categories naturally favor document databases more than others — this chart shows whether models pick up on that signal or apply the same defaults regardless of domain.

In [50]:
cat_df = df.dropna(subset=["category"]).copy()

if len(cat_df) > 0:
    cat_rates = (
        cat_df
        .groupby(["category", "model"])
        .agg(mongodb_pct=("chose_mongodb", lambda s: round(s.mean() * 100, 1)))
        .reset_index()
    )
    fig = px.bar(
        cat_rates,
        x="category",
        y="mongodb_pct",
        color="model",
        barmode="group",
        text="mongodb_pct",
        title="MongoDB Selection Rate by Category",
        labels={"mongodb_pct": "MongoDB chosen (%)", "category": ""},
        template=TEMPLATE,
        width=CHART_WIDTH,
        height=CHART_HEIGHT,
    )
    fig.update_traces(textposition="outside")
    fig.show()
else:
    print("No category metadata available in this dataset.")

---
### Difficulty Breakdown

Do models make different database choices for beginner vs intermediate vs advanced prompts? More complex applications might push models toward more deliberate technology selection, or the added complexity might just reinforce defaults. This chart shows MongoDB selection rate by difficulty level for each model.

In [51]:
diff_rates = (
    df
    .groupby(["difficulty", "model"])
    .agg(mongodb_pct=("chose_mongodb", lambda s: round(s.mean() * 100, 1)))
    .reset_index()
)

fig = px.bar(
    diff_rates,
    x="difficulty",
    y="mongodb_pct",
    color="model",
    barmode="group",
    text="mongodb_pct",
    title="MongoDB Selection Rate by Difficulty",
    labels={"mongodb_pct": "MongoDB chosen (%)", "difficulty": ""},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    category_orders={"difficulty": ["beginner", "intermediate", "advanced"]},
)
fig.update_traces(textposition="outside")
fig.show()

---
### MongoDB Optimal (Ground Truth Lens)

Sections 1–4 measure *what* models choose. This section asks *how often that choice is correct*. Each prompt is labeled with whether MongoDB is actually a good fit (`is_mongodb_optimal`). The class balance shows how many prompts favor MongoDB vs not — framing how easy it is to score well by always or never picking MongoDB. The accuracy chart then shows each model's alignment with ground truth, split by prompt strategy. A model that only improves MongoDB *rate* without improving *accuracy* is just biased, not better.

In [52]:
gt = df.dropna(subset=["is_mongodb_optimal"]).copy()

# Class balance
balance = gt["is_mongodb_optimal"].value_counts(normalize=True).rename("share").reset_index()
balance.columns = ["is_mongodb_optimal", "share"]
balance["pct"] = (balance["share"] * 100).round(1)
print("Ground truth class balance:")
display(balance[["is_mongodb_optimal", "pct"]])

# Alignment per model × experiment
gt["correct"] = gt["chose_mongodb"] == gt["is_mongodb_optimal"]

alignment = (
    gt
    .groupby(["model", "experiment_type"])
    .agg(accuracy=("correct", lambda s: round(s.mean() * 100, 1)))
    .reset_index()
)
alignment["experiment_type"] = alignment["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig = px.bar(
    alignment,
    x="model",
    y="accuracy",
    color="experiment_type",
    barmode="group",
    text="accuracy",
    title="Alignment with Ground Truth (chose MongoDB ⇔ MongoDB optimal)",
    labels={"accuracy": "Accuracy (%)", "experiment_type": "Prompt"},
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.show()

Ground truth class balance:


,is_mongodb_optimal,pct
0,True,50.3
1,False,49.7


---
## Analysis

### Patterns Observed

1. Strong SQL default across all models. Every model tested exhibits a near-total preference for SQL databases (PostgreSQL, SQLite) regardless of the application domain. MongoDB selection rates never exceed 4%, even on prompts specifically designed to favor document databases. This is not a single-model quirk — it's consistent across Claude, Gemini, and GPT.

   This pattern is consistent with broader research on technology homogeneity in LLM-generated code. [Twist et al. (2025), "LLMs Love Python"](https://arxiv.org/html/2503.17181v1) found that across 8 frontier models, Python appears in 90–97% of generated code when no language is specified, and Flask dominates web server generation at 88% despite FastAPI's faster growth. Most critically, models contradict their own recommendations 83% of the time — they recommend appropriate technologies in conversation but default to popular ones when generating code. Temperature variation and chain-of-thought prompting did not shift which technology was selected.

2. The SQL default is likely training-data-driven, not reasoned. The judge frequently classifies choices as `model-default-sql-no-justification` or `orm-or-framework-defaults-to-sql`. Models don't weigh options and conclude SQL is better — they reach for SQLAlchemy/Prisma/TypeORM by reflex, which pulls in PostgreSQL or SQLite as a side effect. The ORM choice is driving the database choice, not the other way around.

   [Wu et al. (2024), "Generative Monoculture"](https://arxiv.org/html/2407.02209v1) provides the theoretical framework for why this happens: LLM-generated code covers a significantly narrower range of solutions than human-written training data. Critically, RLHF alignment makes this concentration *worse*, not better — the preference optimization process further concentrates outputs around dominant patterns. A companion study found that different LLMs are more correlated with each other than with ground truth, meaning switching between GPT, Claude, and Gemini will not solve the default-stack problem because all models converge from overlapping training distributions.

3. Explicit prompting to consider alternatives doesn't overcome the default. The stack-agnostic prompt adds "briefly consider multiple options and explain why you picked the one you did." Despite this, MongoDB rates stayed flat. Models respond to this prompt by *listing* alternatives in prose and then choosing the same SQL stack anyway. The deliberation is performative.

4. Models mention MongoDB but never select it. Under the stack-agnostic prompt, models mention MongoDB in 30–57% of generations — but choose it as the primary database 0% of the time. Models know MongoDB exists and will discuss as an alternative, but never actually commit to it. This is the clearest evidence of performative deliberation: the prompt successfully triggers *discussion* of alternatives without shifting *selection*.

5. Self-reported reasoning diverges systematically from judge assessment. When the judge flags an unjustified default, models' self-reflections rarely agree. Instead, they reframe the choice as intentional — claiming "specific technical requirement" or "relational data needs SQL" for applications that don't obviously require relational modeling. This pattern is consistent across models.

   This aligns with Anthropic's research on unfaithful chain-of-thought reasoning. [Lanham et al. (2023)](https://arxiv.org/abs/2301.13379) demonstrated that as models become larger and more capable, they produce *less* faithful reasoning. [Anthropic's 2025 study, "Reasoning Models Don't Always Say What They Think"](https://arxiv.org/abs/2505.05410), found that when hints changed a model's answer, models verbalized the actual cause less than 20% of the time — instead constructing elaborate alternative justifications. A key detail: unfaithful chain-of-thought explanations were substantially *longer* than faithful ones. Models don't omit explanations when confabulating; they produce more verbose, more detailed alternative reasoning. This directly predicts what we observe: when asked "why PostgreSQL?", models generate lengthy technical justifications rather than surfacing the training-data-driven default.

6. Models don't differentiate based on application fit. MongoDB selection rates are uniformly low regardless of whether the prompt is labeled MongoDB-optimal or not. Models aren't making a poor judgment call — they aren't making a judgment call at all. The SQL default applies indiscriminately.

   Note: the MongoDB-optimal labels were assigned by a single product manager, so this ground truth reflects one person's expert judgment rather than a multi-rater consensus. A skeptical reader may question whether the labels themselves are biased toward MongoDB. Future iterations should validate with multiple labelers or inter-rater agreement.

### Why This Matters

LLMs are the de facto technology advisors for a growing share of new applications. [GitHub's 2025 Octoverse report](https://github.blog/news-insights/octoverse/octoverse-a-new-developer-joins-github-every-second-as-ai-leads-typescript-to-1/) found that 80% of new GitHub developers use Copilot within their first week. These results show that models are systematically steering developers away from MongoDB — not because of reasoned technical assessment, but because of training data bias toward SQL ecosystems.

The commercial impact is already visible. PostgreSQL-based platforms are actively benefiting from LLM defaults: [Supabase reached a $5 billion valuation](https://techcrunch.com/2025/10/03/supabase-nabs-5b-valuation-four-months-after-hitting-2b/) in late 2025, 2.5× its valuation from just four months prior. [TechCrunch reported](https://techcrunch.com/video/supabase-hit-5b-by-turning-down-million-dollar-contracts-heres-why/) that Supabase has become "the back end of choice for the vibe-coding world," with CEO Paul Copplestone betting that "if he sticks to his own product vision, the world will come to him." Vibe coding tools like Replit and Lovable rely on Supabase as their default backend, and 55% of the most recent Y Combinator batch uses it.

No existing code generation benchmark evaluates technology selection. [HumanEval](https://github.com/openai/human-eval) and MBPP are Python-only algorithmic tasks. [SWE-bench](https://www.swebench.com/) operates within existing repositories where technology is already selected. [BigCodeBench](https://huggingface.co/blog/leaderboard-bigcodebench) specifies which libraries to use. [LiveCodeBench](https://livecodebench.github.io/) uses competitive programming with predetermined constraints. The closest existing work is [DevBench](https://arxiv.org/html/2403.08604v1) (Li et al., 2024), which evaluates software design using LLM-as-judge — but its design tasks focus on UML diagrams and file-tree architectures, not technology selection. A [2025 study on serverless function generation](https://arxiv.org/html/2502.02539v1) explicitly notes this gap, stating that architecture-specific tasks and benchmarks for LLMs do not currently exist. The best-in-class effort in an adjacent space is [Amplifying's "What Claude Code Actually Chooses"](https://amplifying.ai/research/claude-code-picks/) (Ong & Vikati, 2026), which measured default tool recommendations across 2,430 Claude Code responses and 20 categories — finding strong convergence patterns including PostgreSQL as the default database (58% of picks) and near-monopoly defaults for GitHub Actions (94%), Stripe (91%), and shadcn/ui (90%). Their methodology — open-ended prompts with no tool names specified — closely mirrors ours.

## Recommendations

### For MongoDB Generally

1. **Engage model providers on training data representation.** The SQL default is a training signal problem. MongoDB documentation, tutorials, and example applications are underrepresented relative to SQL/ORM content in the corpora these models learned from. Increasing high-quality MongoDB content in public training data (docs, blog posts, open-source projects) is the highest-leverage long-term intervention.
2. **Explore prompt-based interventions that actually work.** The stack-agnostic prompt failed. More targeted prompt engineering — e.g. system prompts that specifically instruct models to evaluate MongoDB alongside SQL for document-heavy workloads — may be more effective. The `mongodb_recommended` prompt variant exists in the codebase but was not benchmarked here; running it would test whether explicit MongoDB recommendation in the system prompt changes behavior. However, the literature suggests prompt engineering alone has limited effectiveness for overcoming training-data-driven bias.
3. **Investigate execution feedback loops.** [Huang et al. (2025)](https://arxiv.org/abs/2309.14345) found that test execution feedback reduced biased code generation from 59.88% to 4.79% for GPT-4 — far more effective than prompt engineering. An analogous approach for database selection — e.g. validating schema fit against requirements before committing to a database — may be more promising than trying to change the initial choice.
4. **Treat coding agents as a first-class user persona.** As outlined in the [Coding Agent Persona memo](https://docs.google.com/document/d/1VgWUt5BAILXEju6u7C-qCSAAr4C7m5U86npbewb1bDc), coding agents are no longer just autocomplete — they architect projects and make technology decisions autonomously. MongoDB should optimize for the agent's internal world-model and provide a machine-first experience that removes human-centric barriers like GUIs and manual configuration.
5. **Invest in local-first developer experience via Atlas CLI.** Coding agents have a strong preference for local-first solutions — tools they can install and spin up in their development environment without leaving the terminal. An [Agent Skill](https://agentskills.io/home) for Atlas CLI local development would meet agents where they are: install Atlas CLI, create a local deployment, develop against it, and migrate to cloud-hosted Atlas when ready. This local-to-cloud continuum is Atlas's natural advantage over SQLite (no production path) and managed Postgres (no local story), but it's currently underdocumented and invisible to agents.
6. **Leverage the recency gradient.** Newer models pick newer tools — Amplifying's data shows sharp generational shifts (e.g. Prisma 79% → 0%, replaced by Drizzle in newer Claude models). This means the window for influencing model defaults is open: freshly published MongoDB content, examples, and integrations disproportionately affect newer model training cuts. We **may** be able to influence the next generation of models by targeting the right content at the right time.

### For App Building Benchmarks

1. **Rerun after future model releases.** The recency gradient observed in Amplifying's data suggests model defaults shift across training cuts. This benchmark should be rerun in approximately six months to track whether newer models exhibit different database selection behavior — particularly as MongoDB increases its presence in public training data and developer content.
2. **Evaluate coding agent harnesses, not just raw models.** This benchmark tests base model completions, but developers increasingly interact with models through coding agent harnesses like Cursor, Claude Code, Windsurf, and Google Antigravity. These harnesses add system prompts, tool configurations, and retrieval pipelines that may amplify or counteract the base model's defaults. Measuring database selection behavior at the agent level — where the actual developer experience happens — is the critical next step. [Amplifying.ai](https://amplifying.ai/research/claude-code-picks/) has built rigorous methodology and infrastructure for exactly this kind of agent-level evaluation; we have a meeting scheduled for April 2, 2026 to discuss using their tooling rather than building a parallel system from scratch.